# 05 – Hypothesis Testing II

In the previous notebook, you tested one sample against a known value.

Most real questions are about **comparing two groups**:
- Does a new drug work better than the old one?
- Do male and female customers spend differently?
- Did the A/B test show a real improvement?

And sometimes you need to compare **many groups at once**:
- Do sales differ across all five regions?
- Does average satisfaction vary across product categories?

This notebook covers:
- Two-sample t-test (independent groups)
- Paired t-test (same subjects, before/after)
- Chi-square test (categorical data)
- One-way ANOVA (three or more groups)
- Effect size and the multiple comparisons problem

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import pandas as pd

rng = np.random.default_rng(42)

## 1) Two-Sample t-Test — Comparing Two Independent Groups

**When to use:** You have two separate, independent groups and want to test whether their means differ.

**H₀:** μ₁ = μ₂ (the population means are equal)  
**H₁:** μ₁ ≠ μ₂ (the means are different) — two-tailed

**Example:** Does a new training programme improve employee performance scores?
- Group A: 40 employees on the old programme
- Group B: 42 employees on the new programme

In [ ]:
# Generate performance scores for two training programmes
old_scores = rng.normal(loc=68, scale=14, size=40)   # old programme
new_scores = rng.normal(loc=74, scale=13, size=42)   # new programme

print("=== Two-Sample t-Test: Training Programme Comparison ===")
print()
print(f"Old programme (n={len(old_scores)}): mean={old_scores.mean():.2f}, std={old_scores.std(ddof=1):.2f}")
print(f"New programme (n={len(new_scores)}): mean={new_scores.mean():.2f}, std={new_scores.std(ddof=1):.2f}")
print(f"Observed difference: {new_scores.mean() - old_scores.mean():.2f} points")
print()

# Welch's t-test (equal_var=False) — safer when std may differ between groups
t_stat, p_val = stats.ttest_ind(new_scores, old_scores, equal_var=False)

print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_val:.4f}")
print()
if p_val < 0.05:
    print("Conclusion: REJECT H₀ — statistically significant difference between programmes.")
else:
    print("Conclusion: FAIL TO REJECT H₀ — no statistically significant difference.")

# Effect size: Cohen's d for two samples
pooled_std = np.sqrt((old_scores.var(ddof=1) + new_scores.var(ddof=1)) / 2)
cohen_d = (new_scores.mean() - old_scores.mean()) / pooled_std
size_label = "small" if abs(cohen_d) < 0.3 else "medium" if abs(cohen_d) < 0.6 else "large"
print(f"Cohen's d = {cohen_d:.3f}  ({size_label} effect)")

**Welch's t-test vs Student's t-test:**

Use `equal_var=False` (Welch's) by default. It handles cases where the two groups have different variances. Student's t-test (`equal_var=True`) assumes equal variances — a stricter assumption that's often violated in practice.

Welch's test is slightly more conservative but more robust. Use it unless you have a specific reason to assume equal variances.

In [ ]:
# Visualise the two distributions side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distributions
all_data = np.concatenate([old_scores, new_scores])
bins = np.linspace(all_data.min() - 5, all_data.max() + 5, 25)

axes[0].hist(old_scores, bins=bins, alpha=0.6, color="steelblue",  edgecolor="white", label="Old programme")
axes[0].hist(new_scores, bins=bins, alpha=0.6, color="darkorange", edgecolor="white", label="New programme")
axes[0].axvline(old_scores.mean(), color="steelblue",  linestyle="--", linewidth=2)
axes[0].axvline(new_scores.mean(), color="darkorange", linestyle="--", linewidth=2)
axes[0].set_title("Score Distributions by Programme", fontweight="bold")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(linestyle="--", alpha=0.4)

# Box plots for cleaner comparison
axes[1].boxplot([old_scores, new_scores], labels=["Old", "New"], patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.5),
                medianprops=dict(color="black", linewidth=2))
colours = ["steelblue", "darkorange"]
for patch, c in zip(axes[1].patches, colours):
    patch.set_facecolor(c)
axes[1].set_title(f"Score Comparisont={t_stat:.2f}, p={p_val:.3f}", fontweight="bold")
axes[1].set_ylabel("Score")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/06_stats/ttest2_plot.png", dpi=120)
plt.close()
print("Two-sample comparison plot saved.")

## 2) Paired t-Test — Before and After the Same Subjects

**When to use:** You measure the **same subjects** twice — before and after an intervention. The pairs are linked, so a two-sample t-test would be wrong (it assumes independence between groups).

**H₀:** The mean difference = 0 (intervention had no effect)  
**H₁:** The mean difference ≠ 0

**Example:** Blood pressure measured for 30 patients before and after taking medication. Each patient has a "before" and "after" reading.

In [ ]:
# Paired t-test — blood pressure before and after medication
n_patients = 30
before_bp = rng.normal(loc=145, scale=12, size=n_patients)   # systolic BP before
# Medication reduces BP by 8-12 mmHg on average, with individual variation
effect     = rng.normal(loc=10, scale=6, size=n_patients)
after_bp   = before_bp - effect

print("=== Paired t-Test: Blood Pressure Medication ===")
print()
print(f"n = {n_patients} patients")
print(f"Mean BP before  : {before_bp.mean():.2f} mmHg")
print(f"Mean BP after   : {after_bp.mean():.2f} mmHg")
print(f"Mean reduction  : {(before_bp - after_bp).mean():.2f} mmHg")
print()

# Paired t-test
t_stat_p, p_val_p = stats.ttest_rel(before_bp, after_bp)
print(f"t-statistic  : {t_stat_p:.4f}")
print(f"p-value      : {p_val_p:.4f}")
print()
if p_val_p < 0.05:
    print("Conclusion: Medication significantly reduces blood pressure.")
else:
    print("Conclusion: No significant reduction detected.")

# Compare with (wrong) two-sample test on same data
t_stat_ind, p_val_ind = stats.ttest_ind(before_bp, after_bp, equal_var=False)
print()
print(f"For comparison — wrong two-sample test: p = {p_val_ind:.4f}")
print("Paired test extracts the signal from the noise more efficiently.")

The paired test is more powerful than the two-sample test here because it controls for between-subject variation — each patient serves as their own control.

A patient with naturally high BP will have high readings both before and after. The paired test focuses only on the **change** within each patient, removing this noise.

## 3) Chi-Square Test — Categorical Data

All tests so far have been about numeric means. What if your outcome is categorical?

**Chi-square test of independence:** Tests whether two categorical variables are related.

**Example:** Is the preference for product type (A, B, C) independent of gender?

**H₀:** The variables are independent (no relationship)  
**H₁:** The variables are not independent (there is a relationship)

In [ ]:
# Chi-square test of independence
# Is product preference related to gender?
# Observed frequencies (from a survey of 300 people)

observed = np.array([
    [55, 40, 25],   # Male:   [Laptop, Phone, Tablet]
    [30, 75, 75],   # Female: [Laptop, Phone, Tablet]
])

products = ["Laptop", "Phone", "Tablet"]
genders  = ["Male",   "Female"]

# Show as a table
obs_df = pd.DataFrame(observed, index=genders, columns=products)
print("Observed frequencies:")
print(obs_df)
print(f"Row totals: {obs_df.sum(axis=1).values}   Total: {obs_df.values.sum()}")
print()

# Chi-square test
chi2, p_val, dof, expected = stats.chi2_contingency(observed)

exp_df = pd.DataFrame(expected, index=genders, columns=products).round(1)
print("Expected frequencies (if independent):")
print(exp_df)
print()
print(f"Chi-square statistic : {chi2:.4f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p_val:.6f}")
print()
if p_val < 0.05:
    print("Conclusion: REJECT H₀ — product preference IS associated with gender.")
else:
    print("Conclusion: FAIL TO REJECT H₀ — no significant relationship detected.")

# Cramer's V — effect size for chi-square
n_total = observed.sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(observed.shape) - 1)))
print(f"Cramér's V = {cramers_v:.3f}  ({'small' if cramers_v<0.1 else 'medium' if cramers_v<0.3 else 'large'} effect)")

**How chi-square works:** It compares observed cell counts to what you'd expect if the variables were truly independent. Large differences → large chi-square → small p-value.

**Cramér's V** is the effect size for chi-square tests:
- Near 0 → weak association
- Near 1 → strong association (rarely seen in practice)

## 4) One-Way ANOVA — Comparing Three or More Groups

When you have three or more groups, running multiple t-tests inflates the Type I error rate. If you do 10 t-tests at α = 0.05, you expect one false positive just by chance.

**ANOVA** (Analysis of Variance) tests all groups simultaneously.

**H₀:** All group means are equal (μ₁ = μ₂ = μ₃ = ...)  
**H₁:** At least one group mean is different

**Example:** Does average customer satisfaction differ across five store locations?

**Important:** ANOVA only tells you that *at least one* group differs. It doesn't tell you *which* ones. For that, you need post-hoc tests.

In [ ]:
# ANOVA — customer satisfaction across 5 store locations
north    = rng.normal(loc=3.9, scale=0.8, size=50)
south    = rng.normal(loc=4.2, scale=0.7, size=48)
east     = rng.normal(loc=3.6, scale=0.9, size=52)
west     = rng.normal(loc=4.0, scale=0.8, size=45)
central  = rng.normal(loc=3.5, scale=1.0, size=40)

groups   = [north, south, east, west, central]
names    = ["North","South","East","West","Central"]

print("=== One-Way ANOVA: Customer Satisfaction by Region ===")
print()
for name, grp in zip(names, groups):
    print(f"  {name:<8}: mean={grp.mean():.3f}  std={grp.std(ddof=1):.3f}  n={len(grp)}")
print()

f_stat, p_val = stats.f_oneway(*groups)
print(f"F-statistic : {f_stat:.4f}")
print(f"p-value     : {p_val:.4f}")
print()
if p_val < 0.05:
    print("Conclusion: REJECT H₀ — satisfaction differs significantly across regions.")
    print("Post-hoc tests needed to identify WHICH regions differ.")
else:
    print("Conclusion: No significant difference across regions.")

In [ ]:
# Post-hoc analysis — Tukey's HSD (Honest Significant Difference)
# Compares all pairs while controlling overall Type I error
from scipy.stats import tukey_hsd

all_data = np.concatenate(groups)
group_labels = np.concatenate([[name]*len(grp) for name, grp in zip(names, groups)])

result = tukey_hsd(*groups)

print("Tukey HSD Post-Hoc Comparisons (significant pairs at α=0.05):")
print("-" * 60)
for i in range(len(names)):
    for j in range(i+1, len(names)):
        p = result.pvalue[i, j]
        diff = groups[i].mean() - groups[j].mean()
        sig = "★" if p < 0.05 else ""
        print(f"  {names[i]:<8} vs {names[j]:<8}: diff={diff:+.3f}  p={p:.4f} {sig}")

In [ ]:
# Visualise ANOVA with boxplots
fig, ax = plt.subplots(figsize=(10, 5))

bp = ax.boxplot(groups, labels=names, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
colours = ["#90CAF9","#A5D6A7","#FFCC80","#F48FB1","#CE93D8"]
for patch, colour in zip(bp["boxes"], colours):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.axhline(np.concatenate(groups).mean(), color="gray", linestyle="--",
           linewidth=1.5, label=f"Overall mean = {np.concatenate(groups).mean():.2f}")
ax.set_title(f"Customer Satisfaction by RegionF={f_stat:.2f}, p={p_val:.4f}", fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Satisfaction Score (1–5)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/06_stats/anova_plot.png", dpi=120)
plt.close()
print("ANOVA plot saved.")

**Why not just run pairwise t-tests?** With 5 groups, that's 10 pairs. At α = 0.05, you'd expect 0.5 false positives — 10 × 0.05. That's the **multiple comparisons problem**.

Tukey's HSD adjusts the threshold for each comparison so the overall error rate stays at 5%. Other corrections include Bonferroni (very conservative) and Benjamini-Hochberg (used in genomics and large-scale studies).

## 5) Assumptions — When Tests Are Valid

Every hypothesis test makes assumptions. Violating them can invalidate your results.

**t-Tests assume:**
1. Data is approximately normally distributed (or n is large enough for CLT to apply)
2. Observations are independent
3. For two-sample: the two groups are independent from each other (violated in paired tests)

**Chi-square assumes:**
1. Observations are independent
2. Expected cell counts ≥ 5 in at least 80% of cells

**ANOVA assumes:**
1. Approximately normal distributions within each group
2. Similar variances across groups (homoscedasticity)
3. Independence of observations

In [ ]:
# Test normality with Shapiro-Wilk test
# H₀: data is normally distributed
# H₁: data is not normally distributed

print("=== Normality Check (Shapiro-Wilk) ===")
print()
test_data = {
    "Normal sample (n=50)"    : rng.normal(70, 12, 50),
    "Skewed sample (n=50)"    : rng.exponential(5, 50),
    "Large normal (n=500)"    : rng.normal(70, 12, 500),
}

for label, data in test_data.items():
    stat, p = stats.shapiro(data)
    result = "NORMAL ✓" if p > 0.05 else "NOT normal"
    print(f"  {label:<30}: W={stat:.4f}  p={p:.4f}  → {result}")

print()
print("Note: Shapiro-Wilk is sensitive to large samples.")
print("With n=500, even tiny non-normality is flagged as significant.")
print("For n > 50, CLT usually makes t-tests robust to non-normality anyway.")

In [ ]:
# Test equal variances with Levene's test
# H₀: all groups have equal variance
# H₁: at least one group has a different variance

print("=== Variance Equality (Levene's Test) ===")
print()
stat, p = stats.levene(*groups)
print(f"Levene statistic : {stat:.4f}")
print(f"p-value          : {p:.4f}")
if p > 0.05:
    print("→ No significant difference in variances. Equal variance assumption is met.")
else:
    print("→ Unequal variances detected. Consider Welch's ANOVA or transformation.")

**When assumptions are violated:**

- **Non-normal data + small sample:** Use non-parametric tests (Mann-Whitney U, Kruskal-Wallis)
- **Unequal variances + two groups:** Use Welch's t-test (default in scipy)
- **Unequal variances + many groups:** Use Welch's ANOVA
- **Expected cell counts < 5 in chi-square:** Use Fisher's exact test

## 6) Choosing the Right Test

```
What are you comparing?
│
├─ One group vs a known value
│   └─ One-sample t-test
│
├─ Two groups
│   ├─ Same subjects (before/after)
│   │   └─ Paired t-test
│   └─ Different subjects
│       ├─ Numeric outcome → Two-sample t-test (Welch's)
│       └─ Categorical outcome → Chi-square or Fisher's exact
│
└─ Three or more groups
    ├─ Numeric outcome → One-way ANOVA + Tukey HSD
    └─ Categorical outcome → Chi-square
```

In [ ]:
# Quick reference: all the tests in one place
print("=== Hypothesis Testing Quick Reference ===")
print()
tests = [
    ("One-sample t-test",   "One group vs fixed value",        "stats.ttest_1samp(data, value)"),
    ("Two-sample t-test",   "Two independent groups",          "stats.ttest_ind(g1, g2, equal_var=False)"),
    ("Paired t-test",       "Before/after same subjects",      "stats.ttest_rel(before, after)"),
    ("Chi-square",          "Two categorical variables",       "stats.chi2_contingency(table)"),
    ("One-way ANOVA",       "3+ groups, numeric outcome",      "stats.f_oneway(g1, g2, g3, ...)"),
    ("Tukey HSD",           "Post-hoc after ANOVA",            "stats.tukey_hsd(g1, g2, g3, ...)"),
    ("Shapiro-Wilk",        "Test normality assumption",       "stats.shapiro(data)"),
    ("Levene's test",       "Test equal variance assumption",  "stats.levene(g1, g2, ...)"),
]

print(f"{'Test':<22} {'Use when':<35} {'scipy call'}")
print("-" * 90)
for test, use, call in tests:
    print(f"{test:<22} {use:<35} {call}")

## Summary

- **Two-sample t-test:** Compare means of two independent groups. Use Welch's (equal_var=False) by default.
- **Paired t-test:** Before/after data on the same subjects. More powerful than two-sample.
- **Chi-square:** Test independence of two categorical variables. Check expected counts ≥ 5.
- **One-way ANOVA:** Compare three or more group means simultaneously. Avoids inflated error from multiple t-tests.
- **Post-hoc tests (Tukey HSD):** After ANOVA, identify which specific pairs differ.
- **Check assumptions:** Shapiro-Wilk (normality), Levene's (equal variance)
- **Effect size always:** p-value tells you if an effect is real; effect size tells you if it matters.

Next up: **Overfitting vs Underfitting** — statistical thinking applied directly to machine learning.

## Practice Questions 1

1. A company wants to know if two manufacturing machines produce bolts of the same average diameter. Machine A: n=30, mean=12.1mm, std=0.4mm. Machine B: n=35, mean=11.9mm, std=0.5mm. Run a Welch's two-sample t-test.

2. 25 students took a placement exam, attended a coaching class, and then took the exam again. Generate mock data (before: normal(55, 10, 25), after: before + normal(8, 5, 25)) and run a paired t-test. Compare to an independent two-sample t-test on the same data to see the difference in p-values.

3. Compute Cohen's d for the two-sample test in Q1. Is the effect practically significant?

## Practice Questions 2

1. In a survey of 200 shoppers: 90 males (50 prefer online, 40 prefer in-store), 110 females (80 prefer online, 30 prefer in-store). Run a chi-square test to see if shopping preference depends on gender.

2. A restaurant chain tracks customer wait times across 4 branches. Use ANOVA to test if mean wait times differ across branches (generate your own data). If significant, run Tukey HSD and identify which branches differ.

3. Before running an ANOVA on the wait time data from Q2, check the normality and equal variance assumptions using Shapiro-Wilk and Levene's test.